# Day 2 — Document Parsing & OCR

**Goal:** the cursed PDF pack arrives. Your Day 1 pipeline can't read it.
By EOD, you have a `parse_pdf(path)` function that handles all 4 files.

The pack:
- `01_multipage_table_of_doom.pdf` — table spans 4 pages with repeated headers
- `02_nested_bullet_nightmare.pdf` — 4 levels of nested bullets, page breaks mid-hierarchy
- `03_layout_monster.pdf` — 2-column layout with sidebars
- `04_skewed_scan.pdf` — rotated scan, no text layer, OCR required

## 1. The diagnostic

Before picking a tool, diagnose each PDF. This is the consultant habit we want
to build: **inspect first, tool second**.

Day 2 moves upstream in the pipeline: before retrieval can work, documents must be parsed into text that preserves meaning.

**Core lesson**
- Retrieval quality is often bottlenecked by parsing, not by the vector database or the LLM.
- PDFs are especially tricky because they mix text, layout, scans, tables, and visual structure under one file extension.


In [ ]:
# Start with diagnostics rather than extraction.
# A quick structural survey helps us choose the right parsing strategy per PDF instead of guessing.
from pathlib import Path
import pdfplumber
from pypdf import PdfReader

PACK = Path("./cursed_pack")
pdfs = sorted(PACK.glob("*.pdf"))


def diagnose(path: Path) -> dict:
    diag = {"file": path.name, "size_kb": path.stat().st_size / 1024}

    with pdfplumber.open(path) as pdf:
        diag["pages"] = len(pdf.pages)
        total_chars = 0
        total_tables = 0
        for page in pdf.pages:
            text = page.extract_text() or ""
            total_chars += len(text)
            total_tables += len(page.extract_tables())
        diag["chars_per_page"] = total_chars / diag["pages"] if diag["pages"] else 0
        diag["tables_detected"] = total_tables

    # Heuristic: if chars per page < 50, it's almost certainly a scan
    diag["likely_scan"] = diag["chars_per_page"] < 50
    return diag


for p in pdfs:
    d = diagnose(p)
    print(f"{d['file']}")
    print(f"  pages: {d['pages']}, chars/page: {d['chars_per_page']:.0f}, "
          f"tables: {d['tables_detected']}, likely_scan: {d['likely_scan']}")
    print()


**Expected findings:**
- `01` multi-page table has ~4 tables (one per page — naive parsers double-count)
- `02` nested bullets has 0 tables, all text (layout hierarchy lost in flat extraction)
- `03` layout monster has 1 table (the sidebar box) and columns that interleave
- `04` skewed scan has ~0 chars extractable (no text layer, must OCR)

Diagnostics come first because different PDFs fail in different ways.

**What we are checking**
- Does the file already contain extractable text, or is it basically an image?
- Are tables and layout important to meaning?
- Is a generic parser likely to scramble reading order or miss structure?


## 2. PDF 4 — Skewed scan with Tesseract

The scan has no text layer, so we must OCR. First attempt: run Tesseract on
the pages as-is, see how bad it is, then add deskewing.

OCR is necessary when the PDF is really a scan. A skewed scan is a good example of why raw OCR alone is often not enough.

**Reasoning note**
- Preprocessing is not cosmetic. Deskewing changes the geometry of the page so the OCR model can identify characters more reliably.


In [ ]:
# Render the scan as images because OCR models operate on pixels, not PDF text objects.
# We first capture a naive baseline so we can tell whether preprocessing actually helps.
from pdf2image import convert_from_path
import pytesseract

scan_path = PACK / "04_skewed_scan.pdf"
pages = convert_from_path(str(scan_path), dpi=200)
print(f"Rendered {len(pages)} pages as images")

# Naive OCR — no preprocessing
naive_text = pytesseract.image_to_string(pages[0])
print("=== Naive OCR on page 1 (first 500 chars) ===")
print(naive_text[:500])


### Deskew before OCR

Tesseract hates rotated text. We detect the skew angle via the bounding box of
non-white pixels, then rotate to correct.

Two common methods:
1. **minAreaRect** — find the minimum-area rectangle that contains all text pixels. The rectangle's angle is the skew.
2. **Hough transform** — detect long horizontal lines (text baselines) and measure their angle.

minAreaRect is simpler and works well on full-page text. Use it first.

Deskewing is a classic preprocessing step: estimate the text angle, rotate the image back, then rerun OCR.

**Concept to remember**
- Small geometric errors compound quickly. A document that is only slightly rotated can still lose a lot of OCR quality.


In [ ]:
# Deskewing estimates the dominant text angle and rotates the page back toward horizontal.
# That small geometric correction can make OCR noticeably more readable.
import cv2
import numpy as np


def deskew(pil_img) -> "PIL.Image.Image":
    """Deskew using minAreaRect."""
    from PIL import Image
    img = np.array(pil_img.convert("L"))

    # Invert so text is white (255), background is black (0)
    # Otherwise minAreaRect will fit around the paper instead of the text
    inverted = cv2.bitwise_not(img)
    # Threshold to pure binary
    _, thresh = cv2.threshold(inverted, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    # Get coordinates of all text pixels
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) == 0:
        return pil_img

    # minAreaRect returns (center, (w, h), angle)
    # angle is in [-90, 0]; we need to flip
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    else:
        angle = -angle

    # Rotate the *original* (non-thresholded) image
    h, w = img.shape
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(
        np.array(pil_img), M, (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )
    print(f"  detected skew angle: {angle:.2f}°")
    return Image.fromarray(rotated)


# Deskew and re-OCR page 1
from PIL import Image
deskewed = deskew(pages[0])
deskew_text = pytesseract.image_to_string(deskewed)
print("\n=== Deskewed OCR on page 1 (first 500 chars) ===")
print(deskew_text[:500])


### Measure the improvement

Rough word-count comparison — real evaluation would use word error rate
against ground truth, but for a demo the count delta already tells the story.

Measuring improvement keeps us honest.

**Why we quantify**
- It is easy to feel that preprocessing helped because the output looks cleaner.
- Even a simple proxy like word count helps verify whether the pipeline is actually recovering more usable text.


In [ ]:
# Word count is only a proxy, but it is a fast way to quantify whether preprocessing recovered more usable text.
# In a fuller evaluation, you would also inspect correctness, not just volume.
def word_count(text):
    return len([w for w in text.split() if len(w) > 1])

naive_words = word_count(naive_text)
deskew_words = word_count(deskew_text)
print(f"Naive OCR:    {naive_words} words")
print(f"Deskewed OCR: {deskew_words} words")
print(f"Improvement:  +{deskew_words - naive_words} words "
      f"({100 * (deskew_words - naive_words) / max(naive_words, 1):.0f}%)")


### OCR the full document

Once the approach works on one page, we generalize it to the full document.

**Implementation goal**
- Return a page-aware structure so later retrieval or debugging can still reference which page the text came from.


In [ ]:
# Wrap OCR into a reusable page-level function so downstream code can preserve page provenance.
# Returning structured page records also makes debugging much easier.
def ocr_pdf(path: Path) -> list[dict]:
    pages_img = convert_from_path(str(path), dpi=200)
    out = []
    for i, pg in enumerate(pages_img, 1):
        fixed = deskew(pg)
        text = pytesseract.image_to_string(fixed)
        out.append({"page": i, "text": text})
    return out


scan_pages = ocr_pdf(scan_path)
for p in scan_pages:
    print(f"--- Page {p['page']} ---")
    print(p["text"][:200])
    print()


## 3. PDF 1 — Multi-page table

pdfplumber detects 4 separate tables (one per page) — we want 1 logical table.
Strategy: extract all tables, identify the repeating header, merge bodies, drop
"Subtotal" rows.

Tables are a special parsing problem because meaning depends on row and column relationships, not just raw text.

**Why multi-page tables are tricky**
- Headers may repeat on each page.
- Naive extraction can duplicate rows or lose continuity across page boundaries.
- We need to normalize the structure before turning it into chunks.


In [ ]:
# Multi-page tables need normalization across page boundaries.
# The goal is to keep one logical table, not a pile of disconnected page-level fragments.
table_path = PACK / "01_multipage_table_of_doom.pdf"


def extract_multipage_table(path: Path) -> list[list]:
    all_rows = []
    header = None

    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            for table in page.extract_tables():
                if not table:
                    continue
                # First non-empty table on first page sets the header
                if header is None:
                    header = table[0]
                    body = table[1:]
                else:
                    # Drop header if it repeats
                    body = table[1:] if table[0] == header else table

                # Drop subtotal rows (heuristic: contains "Subtotal" in any cell)
                body = [
                    row for row in body
                    if not any("Subtotal" in str(c) for c in row)
                ]
                all_rows.extend(body)

    return [header] + all_rows


merged = extract_multipage_table(table_path)
print(f"Header: {merged[0]}")
print(f"Data rows: {len(merged) - 1}")
print(f"First data row: {merged[1]}")
print(f"Last data row:  {merged[-1]}")


## 4. PDF 2 — Nested bullets with Docling

Regex won't save you on nested bullets. You need a parser that understands
layout. Docling (IBM, open source) is a good default — it returns a structured
document with preserved hierarchy.

Docling is useful when document structure matters more than literal page geometry.

**Why export to markdown**
- Markdown gives us a portable intermediate form that preserves headings and nested lists well enough for downstream chunking.


In [ ]:
# Docling is useful here because document structure matters more than literal page coordinates.
# Markdown output is a convenient bridge into chunking and retrieval.
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
bullet_path = PACK / "02_nested_bullet_nightmare.pdf"
result = converter.convert(str(bullet_path))
dl_doc = result.document

# Export as markdown — preserves heading levels and list nesting
md_text = dl_doc.export_to_markdown()
print(md_text[:1500])


### Turn Docling output into our Chunk format

We want chunks that carry hierarchy. Docling's markdown already encodes it
via heading levels (`#`, `##`) and list indentation. For chunking, we can
either split on headings (gets us chunks-per-section) or keep the markdown
intact and let a smarter splitter handle it. For now: chunks-per-section.

Chunking markdown at heading boundaries is a pragmatic way to preserve section meaning.

**Benefit**
- A heading path acts like lightweight metadata, which later helps both retrieval and explanation of where a chunk came from.


In [ ]:
# Heading-aware chunking tries to preserve the author's structure instead of slicing blindly.
# The heading stack becomes valuable metadata for both retrieval and explanation.
import re
from utils import Chunk, make_chunk_id


def chunks_from_markdown(md: str, source: str) -> list[Chunk]:
    """Split markdown at heading boundaries; track heading path."""
    lines = md.split("\n")
    chunks: list[Chunk] = []
    current_heading_stack: list[str] = []  # e.g. ["1. Purpose", "2.1 Scope"]
    buffer: list[str] = []
    idx = 0

    def flush():
        nonlocal idx
        if not buffer:
            return
        text = "\n".join(buffer).strip()
        if not text:
            return
        section = " > ".join(current_heading_stack) if current_heading_stack else None
        chunks.append(Chunk(
            id=make_chunk_id(source, idx, text),
            text=text,
            source=source,
            section=section,
            content_type="prose",
        ))
        idx += 1

    for line in lines:
        m = re.match(r"^(#{1,6})\s+(.*)", line)
        if m:
            flush()
            buffer = []
            level = len(m.group(1))
            heading = m.group(2).strip()
            # Truncate stack to this level, push new heading
            current_heading_stack = current_heading_stack[:level - 1] + [heading]
        else:
            buffer.append(line)
    flush()

    return chunks


bullet_chunks = chunks_from_markdown(md_text, bullet_path.name)
print(f"\nExtracted {len(bullet_chunks)} chunks from {bullet_path.name}")
for c in bullet_chunks[:3]:
    print(f"\n  section: {c.section}")
    print(f"  text preview: {c.text[:120]}...")


## 5. PDF 3 — Layout monster

Docling handles column reading order automatically. Compare to pdfplumber's
naive extraction to see what you gain.

Complex layouts are where naive extraction often breaks reading order.

**What to compare**
- Look at whether sentences jump between columns.
- Check whether headings remain attached to the right body text.
- Ask whether the extracted text still reads like a human would read the page.


In [ ]:
# Compare naive extraction with a structure-aware parser on the same file.
# Layout-heavy documents often look 'fine' at a glance while still corrupting reading order.
layout_path = PACK / "03_layout_monster.pdf"

# Naive extraction with pdfplumber
with pdfplumber.open(layout_path) as pdf:
    naive_layout = "\n\n".join(pg.extract_text() or "" for pg in pdf.pages)

print("=== Naive pdfplumber on layout_monster (first 600 chars) ===")
print(naive_layout[:600])
print("\n...notice sentences jumping between columns.\n")

# Docling
result = converter.convert(str(layout_path))
dl_layout_md = result.document.export_to_markdown()
print("=== Docling on layout_monster (first 600 chars) ===")
print(dl_layout_md[:600])


## 6. Putting it together — a unified parse_pdf function

Your deliverable for Day 2 is a function that handles all 4 PDFs by choosing
the right strategy per document.

A unified parser chooses the least-bad strategy per document instead of forcing every PDF through one tool.

**Design principle**
- Real ingestion pipelines branch based on diagnostics: scan detection, table density, layout complexity, or document type.


In [ ]:
# A unified parser branches based on document diagnostics.
# Real-world ingestion pipelines usually need this kind of routing logic to stay reliable.
def parse_pdf(path: Path) -> list[Chunk]:
    """Diagnose and parse any PDF, returning a list of Chunks."""
    diag = diagnose(path)
    source = path.name

    if diag["likely_scan"]:
        # Scanned — OCR with deskewing
        chunks = []
        for page_data in ocr_pdf(path):
            text = page_data["text"].strip()
            if text:
                chunks.append(Chunk(
                    id=make_chunk_id(source, page_data["page"], text),
                    text=text,
                    source=source,
                    page=page_data["page"],
                    content_type="prose",
                    document_type="scanned_contract",
                ))
        return chunks

    if diag["tables_detected"] >= diag["pages"]:
        # Dense tabular — use multipage table extraction
        rows = extract_multipage_table(path)
        # One chunk per data row, carrying the header for context
        header = rows[0]
        chunks = []
        for i, row in enumerate(rows[1:], 1):
            row_text = " | ".join(f"{h}: {v}" for h, v in zip(header, row) if v)
            chunks.append(Chunk(
                id=make_chunk_id(source, i, row_text),
                text=row_text,
                source=source,
                content_type="table",
                document_type="inventory_report",
            ))
        return chunks

    # Default: Docling handles layout + hierarchy
    result = converter.convert(str(path))
    md = result.document.export_to_markdown()
    return chunks_from_markdown(md, source)


# Parse all 4
all_cursed_chunks: list[Chunk] = []
for p in pdfs:
    print(f"Parsing {p.name}...")
    cs = parse_pdf(p)
    print(f"  → {len(cs)} chunks")
    all_cursed_chunks.extend(cs)

print(f"\nTotal chunks across the cursed pack: {len(all_cursed_chunks)}")


## 7. Test retrieval on the cursed pack

Store these chunks in Chroma and see if we can answer questions about the
contract (which required OCR), the SOP (nested bullets), and the newsletter
(layout).

This retrieval test closes the loop from parsing to RAG behavior.

**Why it matters**
- Parsing improvements only count if they make relevant evidence retrievable and usable by the answering step.


In [ ]:
# Re-index the parsed chunks so we can test whether better extraction leads to better retrieval.
# This keeps the lesson grounded in downstream behavior, not just prettier parsing output.
from utils import get_chroma_client, add_chunks
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chroma = get_chroma_client("./chroma_db")

try:
    chroma.delete_collection("day2_cursed")
except Exception:
    pass

coll = chroma.create_collection(name="day2_cursed")
embs = st_model.encode([c.text for c in all_cursed_chunks], show_progress_bar=True)
add_chunks(coll, all_cursed_chunks, embeddings=embs.tolist())
print(f"Indexed {coll.count()} chunks.")


In [ ]:
# Reuse the same embedding model for consistency; the variable we want to test here is parsing quality.
# The answering helper is intentionally simple so extraction issues remain visible.
from utils import get_llm_client

client, generate = get_llm_client()


def embed(texts):
    return st_model.encode(texts).tolist()


def ask_cursed(question: str, k: int = 4) -> str:
    q_emb = embed([question])[0]
    res = coll.query(query_embeddings=[q_emb], n_results=k)
    sources_block = ""
    for i, (text, md) in enumerate(zip(res["documents"][0], res["metadatas"][0]), 1):
        label = f"{md.get('source')}"
        if md.get("section"):
            label += f" ({md['section']})"
        sources_block += f"[Source {i}: {label}]\n{text}\n\n"
    prompt = (
        f"Sources:\n\n{sources_block}\n"
        f"Question: {question}\n\nAnswer using only the sources above."
    )
    return generate(
        system="You are a precise assistant. Answer only from the sources. Be concise.",
        user=prompt,
    )


print("Q: What is the governing law of the Master Services Agreement?")
print("A:", ask_cursed("What is the governing law of the Master Services Agreement?"))
print()
print("Q: What are Class A critical components and how are they inspected?")
print("A:", ask_cursed("What are Class A critical components and how are they inspected?"))
print()
print("Q: How much was SKU-019 worth in total?")
print("A:", ask_cursed("How much was SKU-019 worth in total?"))


## 8. Reflection

- **The diagnostic saved time.** You didn't try OCR on a born-digital PDF or
  throw pdfplumber at a scan. That's the pattern.
- **Deskewing matters.** On our skewed scan, ~30% of words were recovered by
  the rotation fix alone.
- **Docling gave us hierarchy for free.** That `section` field ("II.A.1 > ...")
  will become critical for metadata filtering on Day 3.
- **Tables are still ugly.** We stored one chunk per row, which works for
  lookup queries ("what is SKU-019 worth?") but is bad for aggregations
  ("what's the total of all Pneumatics items?"). Day 3 we'll add metadata to
  enable filtered retrieval.

Reflection here should connect extraction choices to downstream quality.

**Prompt for yourself**
- Which parsing failures would have been invisible if you looked only at answer quality?
- Which document features most strongly predicted the right parser?
